In [1]:
import json, os, logging
import pandas as pd
from tqdm import tqdm

import torch, transformers, sklearn, datasets
from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

device = torch.cuda.current_device() if torch.cuda.is_available() else 'cpu'

0. Utils

In [6]:
def generate_kmer(sequences: list, labels: list, k: int, overlap:int=1, distributed_training:bool=False, chunck_size=100000, logger: logging = logging):
    """
    Generate k-mer string from DNA sequence.
    """
    # utils
    def kmer(sequence: str, k: int, overlap: int):
        return " ".join([sequence[j:j+k] for j in range(0, len(sequence) - k + 1, overlap)])
    
    if not distributed_training:
        return [kmer(sequences[i], k, overlap) for i in range(len(sequences))], labels
    else:
        rank = torch.distributed.get_rank()
        world_size = torch.distributed.get_world_size()

        if rank == 0:
            kmer_seq = [kmer(sequences[i], k, overlap) for i in range(len(sequences))]
        else:
            kmer_seq = None 

        # broadcast k-mer data from rank 0 to all other ranks
        kmer_list = [kmer_seq]
        torch.distributed.broadcast_object_list(kmer_list, src=0)
        kmer_seq = kmer_list[0]  
        
        # partition the data among ranks
        total_size  = len(kmer_seq)
        start_idx   = rank * (total_size // world_size)
        end_idx     = (rank + 1) * (total_size // world_size) if rank != world_size - 1 else total_size
        kmer_subset = kmer_seq[start_idx:end_idx]
        label_subset = labels[start_idx:end_idx]
        logger.warning(f"Rank {rank}: Assigned data indices {start_idx} to {end_idx}")
        return kmer_subset, label_subset
    
def compute_and_display_metrics(predictions, true_labels, class_names=None):
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, average='macro', zero_division=0)
    recall = recall_score(true_labels, predictions, average='macro', zero_division=0)
    f1 = f1_score(true_labels, predictions, average='macro', zero_division=0)
    conf_matrix = confusion_matrix(true_labels, predictions)

    # Prepare metrics dictionary
    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    }
    
    # Print metrics
    print("Performance Metrics:")
    for key, value in metrics.items():
        print(f"{key}: {value:.4f}")
    
    # Display confusion matrix
    print("\nConfusion Matrix:")
    if class_names is None:
        class_names = [str(i) for i in range(len(conf_matrix))]
    
    df_cm = pd.DataFrame(conf_matrix, index=class_names, columns=class_names)
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted Labels")
    plt.ylabel("True Labels")
    plt.title("Confusion Matrix")
    plt.show()
    
    # Print detailed classification report
    print("\nClassification Report:")
    print(classification_report(true_labels, predictions, target_names=class_names))

    return metrics

def prediction(model, tokenizer, sequences):
    batch_size = 640
    predictions = []
    
    for i in tqdm(range(0, len(sequences), batch_size), desc="Batch Prediction"):
        batch_sequences = sequences[i:i+batch_size]
        
        # tokenize the batch
        inputs = tokenizer(
            batch_sequences,
            return_tensors="pt",
            max_length=tokenizer.model_max_length,
            padding="max_length",
            truncation=True
        )
        inputs = {key: val.to(device) for key, val in inputs.items()}
        
        # predictions
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            predicted_classes = torch.argmax(logits, dim=-1).tolist()
        
        # Collect predictions
        predictions.extend(predicted_classes)

    return predictions

1. instanciate the model from checkpoint

In [3]:
checkpoint = "../finetune/output/BaBert_k6-TIS-PseudoLabel/checkpoint-184"
device = torch.cuda.current_device() if torch.cuda.is_available() else 'cpu'

model = transformers.AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    trust_remote_code=True,
).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    checkpoint,
    model_max_length=252,
    padding_side="right",
    use_fast=True,
    trust_remote_code=True,
)

/home/mahouzonssou.akoteno/.local/lib/python3.8/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


/home/mahouzonssou.akoteno/.conda/envs/dnabert/lib/python3.8/site-packages/transformers/modeling_utils.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.loa

3. Load unlabeled data

In [7]:
files = [file for file in os.listdir('../data/tis/BacteriaTIS-ORFSeq2Class-v3/database') if file.endswith('.csv')]

In [ ]:
start_idx = 10
end_idx   = 20
files = files[start_idx:end_idx]
file_path = [os.path.join('../data/tis/BacteriaTIS-ORFSeq2Class-v3/database', file) for file in files]

In [9]:
dfs   = [pd.read_csv(file) for file in file_path]
df    = pd.concat(dfs, ignore_index=True)
df    = df.dropna(subset=["sequence"])
seq   = df['sequence'].values
label = df['label'].values
df.shape

(987811, 2)

In [ ]:
def save_logits_on_unlabeled_data(model, tokenizer, sequences):
    batch_size = 4096 #3072 # 1024*3 --> 80gb gpu / node
    predictions = []
    all_logits  = []
    
    for i in tqdm(range(0, len(sequences), batch_size), desc="Batch Prediction"):
        batch_sequences = sequences[i:i+batch_size]
        
        # tokenize the batch
        inputs = tokenizer(
            batch_sequences,
            return_tensors="pt",
            max_length=tokenizer.model_max_length,
            padding="max_length",
            truncation=True
        )
        inputs = {key: val.to(device) for key, val in inputs.items()}
        
        # predictions
        with torch.no_grad():
            outputs = model(**inputs)
            #logits = outputs.logits
            #predicted_classes = torch.argmax(logits, dim=-1).tolist()
            logits = outputs.logits.cpu().numpy()
        
        # Collect predictions
        #predictions.extend(predicted_classes)
        # Store logits
        all_logits.extend(logits)

    return all_logits

In [8]:
seq_kmer, labels1 = generate_kmer(seq, label, k=6, overlap=1)

In [ ]:
all_logits = save_logits_on_unlabeled_data(model, tokenizer, seq_kmer)

In [ ]:
all_data = []
for seq, logit in zip(seq, all_logits):
    probs = np.exp(logit) / np.sum(np.exp(logit))
    all_data.append([seq] + logit.tolist() + probs.tolist())

#all_data

In [ ]:
df = pd.DataFrame(all_data)
df.to_csv("./temp/pseudo_labeling_2.csv", index=False, header=["sequence", "logit_cls0", "logit_cls1", "prob_cls0", "prob_cls1"])
df.head()

In [ ]:
# DO MANUALLY : APPLY THRESHOLD TO FILTER NEW DATASET

In [ ]:
df = pd.read_csv("./temp/pseudo_labeling_2.csv")
df.head()

,sequence,logit_cls0,logit_cls1,prob_cls0,prob_cls1
0,TCAAAAATTGCACACGAATTGCTGGTAGATCTGGACAAAGATACCG...,1.463655,-1.529449,0.952262,0.047738
1,GATCGCAGCGGCGTGGTTATAGAACCGATGTTGACAGATCAATGGT...,1.397156,-1.517359,0.948559,0.051441
2,CGTGGAGCGCTGAAAAGCATTCTATCTGAACTGGTACGTCAGGATC...,1.483199,-1.535249,0.953401,0.046599
3,GTAAAAAGCGAACAACTAGTCATGTGCGCGCTGAAAAGCGCCACAA...,1.394177,-1.514694,0.948283,0.051717
4,GTTACGCGCATAGTAAACGAAAACCGTCGCTTGATTGTTACCATTG...,1.415485,-1.502432,0.948725,0.051275


In [5]:
df['prob_cls1'].describe()

count    987811.000000
mean          0.053035
std           0.004331
min           0.027166
25%           0.050536
50%           0.052830
75%           0.055228
max           0.339746
Name: prob_cls1, dtype: float64

In [12]:
label

array([0, 0, 0, ..., 0, 0, 0])

In [15]:
np.count_nonzero(label == 1)

24107

In [28]:
# ---

In [47]:
positive_df = df[df['prob_cls1'] > 0.1]
positive_df['label'] = 1
positive_df.shape

/tmp/ipykernel_1656591/1464470627.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_df['label'] = 1


(201, 6)

In [45]:
df_filtered = df[df['prob_cls1'] <= 0.1]
df_highest = df_filtered[df_filtered['prob_cls0'] == df_filtered['prob_cls0'].max()]

top_n = positive_df.shape[0]
df_top_n = df_filtered.nlargest(top_n, 'prob_cls0')
df_top_n['label'] = 0
df_top_n.shape

(2901, 6)

In [46]:
merged_df = pd.concat([positive_df, df_top_n], ignore_index=True)
shuffled_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)
shuffled_df[["sequence", "label"]].to_csv("../data/verified-genome/tis_iter/database/bacteria_pseudo2.csv", index=False)
